In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from phd.Augmentations.common_code.augmentation_functions import *
from phd.Augmentations.common_code.get_dataset import *
from phd.Augmentations.common_code.models import *
from phd.Augmentations.common_code.training_loop import *
from phd.Augmentations.common_code.utils import *
import tensorflow as tf

tf.get_logger().setLevel('ERROR')

In [ ]:
import wandb

wandb.login()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, multiply, Conv2D, Dense, BatchNormalization, Input, MaxPool2D, Dropout, Flatten, Concatenate, add, AveragePooling2D
import numpy as np
from sklearn.preprocessing import normalize
from keras import layers
from keras import ops
from tensorflow.keras.applications import InceptionV3
import keras
from scipy.ndimage import rotate

@keras.saving.register_keras_serializable()
class FocalConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding='same')

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        # based on https://gist.github.com/andrewgiessel/46355632
        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class WaveConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, k_factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.k_factor = k_factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding='same')

    def build(self, input_shape):
        self.wave = self.make_wave(size=input_shape[1], depth=self.filters, k_factor=self.k_factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.wave, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "k_factor": self.k_factor
       }

    # based on http://lampz.tugraz.at/~hadley/physikm/script/waves/circular2d.en.php
    def make_wave(self, size: int, depth: int, k_factor: float = 2):
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
        x0 = y0 = size // 2

        k = np.pi / k_factor
        x, y = x - x0, y - y0
        l = np.sqrt(x * x + y * y)
        u = np.cos(k * l )
        u = normalize(u)
        return 1 - tf.expand_dims(np.stack([u for _ in range(depth)], -1), axis=0)



def BestLoss_4Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size=7, factor=8, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))


  model.add(Conv2D(filters=64, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(WaveConv(filters=64, kernel_size=5, k_factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(224,activation='relu'))
  model.add(Dense(224,activation='relu'))
  model.add(Dense(160,activation='relu'))
  if training_config["NUMBER_OF_CLASSES"] == None:
    model.add(Dense(1, activation='sigmoid'))
  else:
    model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  return model

def Top10Loss_4Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size=7, factor=8, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size=5, factor=5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(WaveConv(filters=32, kernel_size=5, k_factor = 3, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(32,activation='relu'))
  model.add(Dense(160,activation='relu'))
  model.add(Dense(160,activation='relu'))
  if training_config["NUMBER_OF_CLASSES"] == None:
    model.add(Dense(1, activation='sigmoid'))
  else:
    model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  return model


def BestF1_4Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=16, kernel_size=7, factor=7, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=7, factor=6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(224,activation='relu'))
  model.add(Dense(128,activation='relu'))
  model.add(Dense(64,activation='relu'))
  if training_config["NUMBER_OF_CLASSES"] == None:
    model.add(Dense(1, activation='sigmoid'))
  else:
    model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  return model

def Top10F1_4Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=16, kernel_size=7, factor=5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size=7, factor=3, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=7, factor=6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))



  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(192,activation='relu'))
  if training_config["NUMBER_OF_CLASSES"] == None:
    model.add(Dense(1, activation='sigmoid'))
  else:
    model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  inputs = keras.Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))
  outputs = model(inputs)
  built_model = keras.Model(inputs=inputs, outputs=outputs)
  
  return built_model

def BestLoss_11Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=48, kernel_size=3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=7, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=3, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
  return model


def Top10Loss_11Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=48, kernel_size=7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=7, factor = 7, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=5, factor = 8, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=5, factor = 6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(128, activation='relu'))
  model.add(Dense(96, activation='relu'))
  model.add(Dense(64, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
  return model

def BestF1_11Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=16, kernel_size=5, factor = 6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size=3, factor = 9, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(WaveConv(filters=48, kernel_size=7, k_factor = 2, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(192, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def Top10F1_11Class(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=16, kernel_size=7, factor = 6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=5, factor = 6, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size=7, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(128, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model


def Cheng(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=32, kernel_size=3, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size=3, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=128, kernel_size=2, activation='relu'))
  model.add(MaxPool2D(2))


  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
  return model



In [ ]:
from datetime import datetime


tf.get_logger().setLevel('ERROR')

train_size = 43 * 4096 + 2048
test_size = 4 * 4096 + 3371


training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 256,
"TEST_BATCH_SIZE": 256,
"NUMBER_OF_EPOCHS": 500,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": train_size,
"TPU": False,
"AUGMENTATIONS_ROTATE": False,
"AUGMENTATIONS_FLIP_HORIZONTALLY": False,
"AUGMENTATIONS_FLIP_VERTICALLY": False,
"AUGMENTATIONS_ZOOM": False,
"AUGMENTATIONS_SALT_AND_PEPPER": False,
"AUGMENTATIONS_CUTOUT": False,
"AUGMENTATIONS_RANDOM_NOISE": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": train_size,
"TEST_DATASET_SIZE": test_size,
"STRETCH": False,
"STEPS_PER_EXECUTION": 1,
"FOLDS": 10,
"NUMBER_OF_CLASSES": 11,

}
training_config["WANDB_PROJECT_NAME"] = f"Second paper"
training_config["EXPERIMENT_DESCRIPTION"] = f"paper2_multiclass_11_classes_baseline"
training_config["LOCAL_GCP_PATH_BASE"] = f"./training_models"
training_config["REMOTE_GCP_PATH_BASE"] = f"./training_models"
training_config["DATASET_PATH"] = f"../TFDataset/11Classes/training" # or FourClass

models = [
    {'name': 'cavanagh', 'func': Cavanagh, 'starting_fold': 1},
    {'name': 'BestF1', 'func': BestF1_11Class, 'starting_fold': 1},
    {'name': 'Top10F1', 'func': Top10F1_11Class, 'starting_fold': 1},
    {'name': 'BestLoss', 'func': BestLoss_11Class, 'starting_fold': 1},
    {'name': 'Top10Loss', 'func': Top10Loss_11Class, 'starting_fold': 1},
    {'name': 'Cheng', 'func': Cheng, 'starting_fold': 1},
]

perform_training(models, training_config)

In [ ]:
from datetime import datetime


tf.get_logger().setLevel('ERROR')

train_size = 43 * 4096 + 2048
test_size = 4 * 4096 + 3371

training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 256,
"TEST_BATCH_SIZE": 256,
"NUMBER_OF_EPOCHS": 500,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": train_size,
"TPU": False,
"AUGMENTATIONS_ROTATE": True,
"AUGMENTATIONS_FLIP_HORIZONTALLY": True,
"AUGMENTATIONS_FLIP_VERTICALLY": True,
"AUGMENTATIONS_ZOOM": True,
"AUGMENTATIONS_RANDOM_NOISE": True,

"AUGMENTATIONS_SALT_AND_PEPPER": False,
"AUGMENTATIONS_CUTOUT": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,             
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": train_size,
"TEST_DATASET_SIZE": test_size,
"STRETCH": False,
"STEPS_PER_EXECUTION": 1,
"FOLDS": 10,
"NUMBER_OF_CLASSES": 11
}

training_config["WANDB_PROJECT_NAME"] = f"Second paper"
training_config["EXPERIMENT_DESCRIPTION"] = f"paper2_multiclass_11_classes_basic_augmentations_with_noise"
training_config["LOCAL_GCP_PATH_BASE"] = f"./training_models"
training_config["REMOTE_GCP_PATH_BASE"] = f"./training_models"
training_config["DATASET_PATH"] = f"../TFDataset/11Classes/training" # or FourClass

models = [
    {'name': 'cavanagh', 'func': Cavanagh, 'starting_fold': 1},
    {'name': 'BestF1', 'func': BestF1_11Class, 'starting_fold': 1},
    {'name': 'Top10F1', 'func': Top10F1_11Class, 'starting_fold': 1},
    {'name': 'BestLoss', 'func': BestLoss_11Class, 'starting_fold': 1},
    {'name': 'Top10Loss', 'func': Top10Loss_11Class, 'starting_fold': 1},
     {'name': 'Cheng', 'func': Cheng, 'starting_fold': 1},
    
]

perform_training(models, training_config)

# Model serialization test

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, multiply, Conv2D, Dense, BatchNormalization, Input, MaxPool2D, Dropout, Flatten
import numpy as np
from sklearn.preprocessing import normalize
from keras import layers
from keras import ops
import keras


train_size = 20 * 4096 + 2048
test_size = 2 * 4096 + 1088

training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 1024,
"TEST_BATCH_SIZE": 1024,
"NUMBER_OF_EPOCHS": 1,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": train_size,
"TPU": False,
"AUGMENTATIONS_ROTATE": True,
"AUGMENTATIONS_FLIP_HORIZONTALLY": True,
"AUGMENTATIONS_FLIP_VERTICALLY": True,
"AUGMENTATIONS_ZOOM": True,
"AUGMENTATIONS_SALT_AND_PEPPER": False,
"AUGMENTATIONS_CUTOUT": False,
"AUGMENTATIONS_RANDOM_NOISE": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": train_size,
"TEST_DATASET_SIZE": test_size,
"STRETCH": False,
"STEPS_PER_EXECUTION": 4,
"FOLDS": 1,
"NUMBER_OF_CLASSES": 4
}
training_config["WANDB_PROJECT_NAME"] = f"Second paper"
training_config["EXPERIMENT_DESCRIPTION"] = f"paper2_4_classes_basic_augmentations"
training_config["LOCAL_GCP_PATH_BASE"] = f"./training_models"
training_config["REMOTE_GCP_PATH_BASE"] = f"./training_models"
training_config["DATASET_PATH"] = f"../TFDataset/FourClasses/training"

strategy = reset_tpu(training_config)




def SimpleTest3(training_config):
#   m = Sequential(name="Cavanagh")
#   m.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   m.add(Conv2D(filters=16, kernel_size=7, activation='relu'))
#   m.add(FocalConv(filters=16, kernel_size=7, activation='relu', factor=3))
#   m.add(WaveConv(filters=16, kernel_size=7, activation='relu', k_factor=3))

#   m.add(Dropout(0.5))

#   m.add(Flatten())

#   m.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

#   return m
    return create_new_vit_classifier(training_config)

def create_model(training_config):
    return SimpleTest3(training_config)

with strategy.scope():

    model_name = "new_vit_test"
    initial_model_path = f"models/128x128/initial_models/4_Classes/{model_name}"



    if not os.path.isdir(f"{training_config['LOCAL_GCP_PATH_BASE']}/{initial_model_path}"):
        os.mkdir(f"{training_config['LOCAL_GCP_PATH_BASE']}/{initial_model_path}")

        optimizer = optimizers.Adam(learning_rate= training_config["LEARNING_RATE"], beta_1=0.9, beta_2=0.999, epsilon=1e-08)
        model = create_model(training_config)

        model.compile(
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            steps_per_execution = 1,
            optimizer=optimizer,
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

        model.save(f"{training_config['REMOTE_GCP_PATH_BASE']}/{initial_model_path}/{model_name}.keras")

        model = tf.keras.models.load_model(f"{training_config['REMOTE_GCP_PATH_BASE']}/{initial_model_path}/{model_name}.keras")
        model.save_weights(f"{training_config['LOCAL_GCP_PATH_BASE']}/{initial_model_path}/{model_name}.weights.h5")

    optimizer = optimizers.Adam(learning_rate= training_config["LEARNING_RATE"], beta_1=0.9, beta_2=0.999, epsilon=1e-08)

    test_dataset = get_intial_fold_dataset(
        training_config,
        f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/fold_1/test",
        seed = training_config["SEED"],
        shuffle = False,
        cache = True).batch(training_config["TEST_BATCH_SIZE"], drop_remainder = training_config["TPU"])

    if training_config["TPU"]:
        test_dataset = test_dataset.prefetch(AUTO)

    cached_initial_training_dataset = get_intial_fold_dataset(
        training_config,
        f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/fold_1/train",
        training_config["SEED"],
        shuffle = True,
        cache = True)

    

    model = create_model(training_config)
    model.load_weights(f"{training_config['LOCAL_GCP_PATH_BASE']}/{initial_model_path}/{model_name}.weights.h5")

    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        steps_per_execution = training_config['STEPS_PER_EXECUTION'],
        optimizer=optimizer,
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

    train_dataset = shuffle_dataset(
        cached_initial_training_dataset,
        training_config,
        training_config["TRAIN_BATCH_SIZE"],
        seed = training_config["SEED"] +1,
        augment = True,
        drop_remainder = False)

    history = model.fit(
        x= train_dataset,
        validation_data = test_dataset,
        epochs = 1,
        steps_per_epoch = training_config["TRAIN_DATASET_SIZE"] // training_config["TRAIN_BATCH_SIZE"],
        validation_steps = training_config["TEST_DATASET_SIZE"] // training_config["TEST_BATCH_SIZE"],
        verbose = 1,
        shuffle = False)

    model.save(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_name}.keras")
    model.save_weights(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_name}.weights.h5")



tf.config.run_functions_eagerly(True)

dataset = get_dataset_with_objids(f"{training_config['LOCAL_GCP_PATH_BASE']}/../TFDataset/FourClasses/training/fold_1/test", 16)

f_model = tf.keras.models.load_model(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_name}.keras")

w_model = create_model(training_config)
w_model.load_weights(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_name}.weights.h5")


for images, labels, objids in dataset:
    m_call = model.call(images, training=False)
    m_pred = model.predict(images)

    f_call = f_model.call(images, training=False)
    f_pred = f_model.predict(images)

    w_call = w_model(images, training=False)
    w_pred = w_model.predict(images)

    print(tf.math.reduce_all(tf.math.equal(m_call, m_pred)))
    print(tf.math.reduce_all(tf.math.equal(w_call, w_pred)))
    print(tf.math.reduce_all(tf.math.equal(f_call, f_pred)))
    print(tf.math.reduce_all(tf.math.equal(m_call, w_call)))
    print(tf.math.reduce_all(tf.math.equal(m_call, f_call)))
    print(tf.math.reduce_all(tf.math.equal(m_pred, w_pred)))
    print(tf.math.reduce_all(tf.math.equal(m_pred, f_pred)))

    break

# Other experiments/architectures

In [ ]:
# def BestF1(training_config):
#   model = Sequential(name="Cavanagh")
#   model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   model.add(Conv2D(filters=48, kernel_size=3, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))


#   model.add(WaveConv(filters=32, kernel_size=5, k_factor = 5, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=64, kernel_size=3, factor = 8, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(Flatten())
#   if training_config["ENABLE_DETERMINISM"]:
#     model.add(Dropout(0.5, seed = training_config["SEED"]))
#   else:
#     model.add(Dropout(0.5))

#   model.add(Dense(192,activation='relu'))
#   model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

#   return model

# def Top10F1(training_config):
#   model = Sequential(name="Cavanagh")
#   model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   model.add(Conv2D(filters=48, kernel_size=3, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))


#   model.add(FocalConv(filters=64, kernel_size=5, factor = 6, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=64, kernel_size=3, factor = 7, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(Flatten())
#   if training_config["ENABLE_DETERMINISM"]:
#     model.add(Dropout(0.5, seed = training_config["SEED"]))
#   else:
#     model.add(Dropout(0.5))

#   model.add(Dense(64, activation='relu'))
#   model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
#   return model

# def BestLoss(training_config):
#   model = Sequential(name="Cavanagh")
#   model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   model.add(FocalConv(filters=16, kernel_size=3, factor = 8, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=48, kernel_size=7, factor = 8, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=64, kernel_size=3, factor = 5, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=48, kernel_size=7, factor = 4, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(Flatten())
#   if training_config["ENABLE_DETERMINISM"]:
#     model.add(Dropout(0.5, seed = training_config["SEED"]))
#   else:
#     model.add(Dropout(0.5))

#   model.add(Dense(96, activation='relu'))
#   model.add(Dense(64, activation='relu'))
#   model.add(Dense(64, activation='relu'))
#   model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
#   return model

# def Top10Loss(training_config):
#   model = Sequential(name="Cavanagh")
#   model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   model.add(FocalConv(filters=16, kernel_size=7, factor = 6, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=48, kernel_size=3, factor = 8, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=64, kernel_size=5, factor = 8, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(Flatten())
#   if training_config["ENABLE_DETERMINISM"]:
#     model.add(Dropout(0.5, seed = training_config["SEED"]))
#   else:
#     model.add(Dropout(0.5))

#   model.add(Dense(32, activation='relu'))
#   model.add(Dense(96, activation='relu'))
#   model.add(Dense(128, activation='relu'))
#   model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
#   return model

# def BestLossWithAugmentations(training_config):
#   model = Sequential(name="Cavanagh")
#   model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   model.add(Conv2D(filters=32, kernel_size=5, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=64, kernel_size=5, factor = 6, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(WaveConv(filters=32, kernel_size=7, k_factor = 9, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(WaveConv(filters=64, kernel_size=5, k_factor = 6, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(Flatten())
#   if training_config["ENABLE_DETERMINISM"]:
#     model.add(Dropout(0.5, seed = training_config["SEED"]))
#   else:
#     model.add(Dropout(0.5))

#   model.add(Dense(160, activation='relu'))
#   model.add(Dense(192, activation='relu'))
#   model.add(Dense(256, activation='relu'))
#   model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
#   return model

# def BestLossFirstLayer(training_config):
#   model = Sequential(name="Cavanagh")
#   model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

#   model.add(FocalLayer(factor=3, filters=3))

#   model.add(Conv2D(filters=32, kernel_size=5, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(FocalConv(filters=64, kernel_size=5, factor = 6, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(WaveConv(filters=32, kernel_size=7, k_factor = 9, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(WaveConv(filters=64, kernel_size=5, k_factor = 6, activation='relu'))
#   model.add(BatchNormalization())
#   model.add(MaxPool2D(2))

#   model.add(Flatten())
#   if training_config["ENABLE_DETERMINISM"]:
#     model.add(Dropout(0.5, seed = training_config["SEED"]))
#   else:
#     model.add(Dropout(0.5))

#   model.add(Dense(160, activation='relu'))
#   model.add(Dense(192, activation='relu'))
#   model.add(Dense(256, activation='relu'))
#   model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
#   return model

def BestLossWithAugmentations(training_config):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=64, kernel_size=3, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size=3, factor = 1, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size=3, factor = 1, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=7, factor = 4, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  if training_config["ENABLE_DETERMINISM"]:
    model.add(Dropout(0.5, seed = training_config["SEED"]))
  else:
    model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(96, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))
  
  return model


@keras.saving.register_keras_serializable()
class FocalLayer(keras.layers.Layer):
    def __init__(self, filters, factor):
        super().__init__()
        self.filters = filters
        self.factor = factor

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class WaveLayer(keras.layers.Layer):
    def __init__(self, filters, k_factor):
        super().__init__()
        self.filters = filters
        self.k_factor = k_factor

    def build(self, input_shape):
        self.wave = self.make_wave(size=input_shape[1], depth=self.filters, k_factor=self.k_factor)

    def call(self, x, training=False):
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.wave, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "k_factor": self.k_factor
       }

    def make_wave(self, size: int, depth: int, k_factor: float = 2):
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
        x0 = y0 = size // 2

        k = np.pi / k_factor
        x, y = x - x0, y - y0
        l = np.sqrt(x * x + y * y)
        u = np.cos(k * l )
        u = normalize(u)
        return 1 - tf.expand_dims(np.stack([u for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class BarLayer(keras.layers.Layer):
    def __init__(self, filters, angle, alpha):
        super().__init__()
        self.filters = filters
        self.angle = angle
        self.alpha = alpha

    def build(self, input_shape):
        self.bar = self.make_bar(input_shape[1], self.alpha, self.angle, self.filters)

    def call(self, x, training=False):
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.bar, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "alpha": self.alpha,
          "angle": self.angle
       }

    def make_bar(self, size: int, alpha: int, angle: int, depth: int):
        arr = np.zeros((size, size))

        bar_size = size // 10
        padding = (size - bar_size) // 2

        for i in range(padding):
            arr[i, :] = ((float(i) / padding) ** alpha)

        arr[padding: padding+bar_size, :] = 1

        for j in range(padding):
            arr[padding + bar_size + j, :] = (-1 * (float(j) / padding) + 1) ** alpha

        arr = rotate(arr, angle, reshape = False, mode='nearest')   
        return tf.expand_dims(np.stack([arr for _ in range(depth)], axis = -1), axis=0)

def Paper3Parallel(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal = FocalLayer(filters=3, factor = 6)(inputs)
  wave1 = WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3 = WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5 = WaveLayer(filters=3, k_factor = 5)(inputs)
  bars = []
  for i in range(0, 180, 45):
    bars.append(BarLayer(alpha = 7, angle = i,  filters = 3)(inputs))

  x = add([focal, wave1, wave3, wave5, *bars])

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 6, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=48, kernel_size=7, factor = 5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  x = Dense(160, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model




@keras.saving.register_keras_serializable(package="mlp", name="mlp")
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=keras.activations.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

@keras.saving.register_keras_serializable(package="Patches")
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size
        self.focal = FocalLayer(filters=3, factor = 6)
        self.wave1 = WaveLayer(filters=3, k_factor = 1)
        self.wave3 = WaveLayer(filters=3, k_factor = 3)
        self.wave5 = WaveLayer(filters=3, k_factor = 5)
        self.bar0 = BarLayer(alpha = 7, angle = 0,  filters = 3)
        self.bar45 = BarLayer(alpha = 7, angle = 45,  filters = 3)
        self.bar90 = BarLayer(alpha = 7, angle = 90,  filters = 3)
        self.bar135 = BarLayer(alpha = 7, angle = 135,  filters = 3)

    def call(self, images):
        input_shape = ops.shape(images)
        batch_size = input_shape[0]
        height = input_shape[1]
        width = input_shape[2]
        channels = input_shape[3]
        num_patches_h = height // self.patch_size
        num_patches_w = width // self.patch_size

        focal_o = self.focal(images)
        wave1_o = self.wave1(images)
        wave3_o = self.wave3(images)
        wave5_o = self.wave5(images)
        bar0_o = self.bar0(images)
        bar45_o = self.bar45(images)
        bar90_o = self.bar90(images)
        bar135_o = self.bar135(images)

        patches = Concatenate()([focal_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
        patches = ops.reshape(
            patches,
            (
                batch_size,
                # num_patches_h * num_patches_w,
                8,
                self.patch_size * self.patch_size * channels,
            ),
        )
        return patches

    def get_config(self):
        return {"patch_size": self.patch_size}

@keras.saving.register_keras_serializable(package="PatchEncoder")
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=self.projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=self.num_patches, output_dim=self.projection_dim
        )

    def call(self, patch):
        positions = ops.expand_dims(
            ops.arange(start=0, stop=self.num_patches, step=1), axis=0
        )
        print("patch shape", patch.shape)
        projected_patches = self.projection(patch)
        encoded = projected_patches + self.position_embedding(positions)
        return encoded

    def get_config(self):
        return {"num_patches": self.num_patches, "projection_dim": self.projection_dim}

def Paper3TRansformerV3(training_config):
    patch_size = 128
    projection_dim = 16
    transformer_layers = 2
    num_heads = 4
    head_size = 1024
    dropout_rate = 0.5

    transformer_units = [
        projection_dim * 2,
        projection_dim,
    ]
    mlp_head_units = [
        head_size * 2,
        head_size,
    ]


    inputs = keras.Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))
    
    # focal = FocalLayer(filters=3, factor = 6)(inputs)
    # wave1 = WaveLayer(filters=3, k_factor = 1)(inputs)
    # wave3 = WaveLayer(filters=3, k_factor = 3)(inputs)
    # wave5 = WaveLayer(filters=3, k_factor = 5)(inputs)
    # bars = []
    # for i in range(0, 180, 45):
    #   bars.append(BarLayer(alpha = 7, angle = i,  filters = 3)(inputs))

    # num_patches = len([focal, wave1, wave3, wave5, *bars])

    # patches = Patches(patch_size)(inputs)

    focal1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
    focal3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
    focal5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
    wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
    wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
    wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
    bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
    bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
    bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
    bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

    patches = Concatenate()([inputs, focal1_o, focal3_o, focal5_o,  wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
    # patches = AveragePooling2D(2)(patches)
    patches = ops.reshape(
        patches,
        (
            256,
            # num_patches_h * num_patches_w,
            11,
            64 * 64 * 3,
        ),
    )

    num_patches = 11
    # Create patches.
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=dropout_rate)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(dropout_rate)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=dropout_rate)
    # Classify outputs.
    logits = layers.Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
            loss=keras.losses.SparseCategoricalCrossentropy(),
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

    return model


def Paper3Conv(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  x = Dense(160, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3ConvV4Gelu(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  # if training_config["ENABLE_DETERMINISM"]:
  #   x = Dropout(0.5, seed = training_config["SEED"])(x)
  # else:
  #   x = Dropout(0.5)(x)

  x = Dense(128, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(160, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(64, activation='gelu')(x)
  x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3ConvV4Gelu(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  # if training_config["ENABLE_DETERMINISM"]:
  #   x = Dropout(0.5, seed = training_config["SEED"])(x)
  # else:
  #   x = Dropout(0.5)(x)

  x = Dense(128, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(160, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(64, activation='gelu')(x)
  x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3ConvV5(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  # if training_config["ENABLE_DETERMINISM"]:
  #   x = Dropout(0.5, seed = training_config["SEED"])(x)
  # else:
  #   x = Dropout(0.5)(x)

  x = Dense(128, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(160, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(64, activation='gelu')(x)
  x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3ConvV5AvgPool(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  # if training_config["ENABLE_DETERMINISM"]:
  #   x = Dropout(0.5, seed = training_config["SEED"])(x)
  # else:
  #   x = Dropout(0.5)(x)

  x = Dense(128, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(160, activation='gelu')(x)
  x = Dropout(0.5)(x)
  x = Dense(64, activation='gelu')(x)
  x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3ConvV5AvgPoolNoDropRelu(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(160, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(64, activation='relu')(x)
  # x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model


# 61.7%
def Paper3ConvV6(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(160, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(64, activation='relu')(x)
  # x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3ConvV7(training_config):
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)

  x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(160, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(64, activation='relu')(x)
  # x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model


def dense_layer(x, output_size = 16):
  average_pool = 4
  x = AveragePooling2D(average_pool)(x)
  x = Dense(32, activation='relu')(x)
  x = Dropout(0.5)(x)
  x = Dense(64, activation='relu')(x)
  x = Dropout(0.5)(x)
  x = Dense(output_size, activation='relu')(x)

  return x

def Paper3ConvDenseV2(training_config):
  average_pool = 4
  num_dense = 2
  
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))



  inputs_reduced = AveragePooling2D(average_pool)(inputs)

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_1_o  =  dense_layer(focal_1_o)


  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_3_o  =  dense_layer(focal_3_o)

  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  focal_5_o  =  dense_layer(focal_5_o)

  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave1_o  =  dense_layer(wave1_o)

  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave3_o  =  dense_layer(wave3_o)

  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  wave5_o  =  dense_layer(wave5_o)

  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar0_o  =  dense_layer(bar0_o)

  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar45_o  =  dense_layer(bar45_o)

  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar90_o  =  dense_layer(bar90_o)

  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)
  bar135_o  =  dense_layer(bar135_o)

  x = Concatenate()([inputs_reduced, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)

  x = Flatten()(x)
  # x = flattened

  # for i in range(2):
  #     x = Dense(32, activation='relu')(x)
  #     # x = Dropout(0.5)(x)
  #     x = Dense(64, activation='relu')(x)
  #     # x = Dropout(0.5)(x)
  #     x = Dense(11, activation='relu')(x)

  #     x = add([x, flattened])

  # x = Conv2D(filters=16, kernel_size=7, activation='relu')(x)
  # x = BatchNormalization()(x)
  # x = MaxPool2D(2)(x)

  # x = Conv2D(filters=32, kernel_size=5, activation='relu')(x)
  # x = BatchNormalization()(x)
  # x = MaxPool2D(2)(x)

  # x = Conv2D(filters=48, kernel_size=7, activation='relu')(x)
  # x = BatchNormalization()(x)
  # x = MaxPool2D(2)(x)

  # x = Flatten()(x)
  # if training_config["ENABLE_DETERMINISM"]:
  #   x = Dropout(0.5, seed = training_config["SEED"])(x)
  # else:
  #   x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(160, activation='relu')(x)
  # x = Dropout(0.5)(x)
  x = Dense(64, activation='relu')(x)
  # x = Dropout(0.5)(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model





# 63.3
def Paper3_OldTop10Loss(training_config):
  
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])

  x = FocalConv(filters=48, kernel_size=7, factor = 10, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=7, factor = 7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 8, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 6, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  x = Dense(96, activation='relu')(x)
  x = Dense(64, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

# 57
def Paper3_OldTop10Loss_Pooling(training_config):
  
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([inputs, focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  x = AveragePooling2D(2)(x)

  x = FocalConv(filters=48, kernel_size=7, factor = 10, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=7, factor = 7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 8, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 6, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  x = Dense(96, activation='relu')(x)
  x = Dense(64, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model

def Paper3_OldTop10Loss_NoPooling_NoImage(training_config):
  
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)

  x = FocalConv(filters=48, kernel_size=7, factor = 10, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=7, factor = 7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 8, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = FocalConv(filters=32, kernel_size=5, factor = 6, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(128, activation='relu')(x)
  x = Dense(96, activation='relu')(x)
  x = Dense(64, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model


def Paper3_NewBestLoss(training_config):
  
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)

  x = FocalConv(filters=32, kernel_size=7, factor = 10, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=48, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=64, kernel_size=5, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Conv2D(filters=64, kernel_size=7, activation='relu')(x)
  x = BatchNormalization()(x)
  x = MaxPool2D(2)(x)

  x = Flatten()(x)
  if training_config["ENABLE_DETERMINISM"]:
    x = Dropout(0.5, seed = training_config["SEED"])(x)
  else:
    x = Dropout(0.5)(x)

  x = Dense(32, activation='relu')(x)
  x = Dense(192, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  return model



@keras.saving.register_keras_serializable(package="REshape")
class Reshape(layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, patch):
      input_shape = ops.shape(patch)
      batch_size = input_shape[0]
      return ops.reshape(
          patch,
          (
              batch_size,
              # num_patches_h * num_patches_w,
              11,
              128 * 128 * 3,
          ),
      )

    def get_config(self):
        return {}

def Paper3TRansformer_BestLoss(training_config):
    patch_size = 128
    projection_dim = 24
    transformer_layers = 4
    num_heads = 3
    head_size = 768
    dropout_rate = 0.5

    transformer_units = [
        projection_dim * 2,
        projection_dim,
    ]
    mlp_head_units = [
        head_size * 2,
        head_size,
    ]


    inputs = keras.Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

    focal1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
    focal3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
    focal5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
    wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
    wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
    wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
    bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
    bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
    bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
    bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

    patches = Concatenate()([inputs, focal1_o, focal3_o, focal5_o,  wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
    # patches = AveragePooling2D(2)(patches)
    patches = Reshape()(patches)

    num_patches = 11
    # Create patches.
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=dropout_rate)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(dropout_rate)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=dropout_rate)
    # Classify outputs.
    logits = layers.Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)

    return model


def Paper3_NewTransformer_BestLoss(training_config):
    patch_size = 128
    projection_dim = 64
    transformer_layers = 4
    num_heads = 4
    head_size = 384
    dropout_rate = 0.5

    transformer_units = [
        projection_dim * 2,
        projection_dim,
    ]
    mlp_head_units = [
        head_size * 2,
        head_size,
    ]


    inputs = keras.Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

    focal1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
    focal3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
    focal5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
    wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
    wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
    wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
    bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
    bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
    bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
    bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

    patches = Concatenate()([inputs, focal1_o, focal3_o, focal5_o,  wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
    # patches = AveragePooling2D(2)(patches)
    patches = Reshape()(patches)

    num_patches = 11
    # Create patches.
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=dropout_rate)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(dropout_rate)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=dropout_rate)
    # Classify outputs.
    logits = layers.Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)

    return model